# MiBici: monitoreo de P1 y análisis complementario de viajes

El reporte principal compara las 16 entradas de P1 por estación y hora contra sus filas elegibles de entrenamiento, de octubre de 2025 a junio de 2026.
El periodo actual es octubre de 2026. Se conserva además el análisis de viajes individuales para estudiar calidad y cambios en la red.

La preparación original debe estar disponible y coincidir con el manifiesto. Instalar las dependencias del proyecto antes de ejecutar.
Los datos y el manifiesto congelados se utilizan como entradas; el CSV nuevo se descarga a data/raw y los reportes se guardan en reports/drift.
Los campos personales se excluyen al cargar el CSV actual. Las cifras reales se obtienen al ejecutar y quedan en el resumen JSON.

[Guía de periodos, población y validación](../references/monitoreo-p1.md).
Guardar este notebook con las salidas limpias antes de subirlo.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from evidently import DataDefinition, Dataset, Report
from evidently.presets import DataDriftPreset, DataSummaryPreset

candidatos = [p for base in [Path.cwd(), *Path.cwd().parents]
              for p in (base, base / 'tec-mlops-equipo-27')]
root = next(p for p in candidatos if (p / 'mibici' / 'config.py').exists())
sys.path.insert(0, str(root))
from mibici import drift
from mibici.config import (
    MES_NUEVO, REPORTS_DIR, MUESTRA_MONITOREO_P1,
    UMBRAL_MONITOREO_P1, PROPORCION_MONITOREO_P1,
)

# --- Parámetros del análisis -------------------------------------------------
MUESTRA = MUESTRA_MONITOREO_P1
UMBRAL_DRIFT = UMBRAL_MONITOREO_P1
PROPORCION_DRIFT = PROPORCION_MONITOREO_P1
MES_CONTROL = f"{int(MES_NUEVO[:4]) - 1}-{MES_NUEVO[5:]}"   # mismo mes del año anterior
DIR_REPORTES = REPORTS_DIR / 'drift'
DIR_REPORTES.mkdir(parents=True, exist_ok=True)
print('Mes nuevo:', MES_NUEVO, '| mes de control (misma estacionalidad):', MES_CONTROL)

## Reporte principal: entradas de P1
Reconstruye la referencia con el pipeline original, verifica el conteo del run documentado y genera los reportes por estación y hora.


In [ ]:
from mibici.monitoreo_p1 import ejecutar

resumen_p1 = ejecutar(
    mes=MES_NUEVO, muestra=MUESTRA,
    umbral=UMBRAL_DRIFT, proporcion=PROPORCION_DRIFT,
)
display(pd.DataFrame(resumen_p1['drift_por_variable']).set_index('variable'))


## 1. Datos: mes nuevo y periodo de referencia
El mes nuevo se descarga con `dataset.descargar` y se limpia con `dataset.normalizar_viajes`, es decir, con los mismos nombres de columna, codificación, tipos y reglas de descarte que el periodo congelado. Si `data/raw/datos_abiertos_2026_10.csv` ya existe, no se vuelve a descargar.

In [ ]:
drift.descargar_mes(MES_NUEVO)
nuevo, stats_nuevo = drift.cargar_mes(MES_NUEVO)
display(pd.Series(stats_nuevo, name=MES_NUEVO).to_frame())

In [ ]:
viajes_ref = drift.cargar_referencia()
estaciones = drift.cargar_nomenclatura()
mes_ref = viajes_ref['inicio_del_viaje'].dt.strftime('%Y-%m')
print('Referencia:', f"{len(viajes_ref):,} viajes,", mes_ref.min(), 'a', mes_ref.max())
print('Actual    :', f"{len(nuevo):,} viajes,", nuevo['inicio_del_viaje'].min().date(), 'a', nuevo['inicio_del_viaje'].max().date())

# Variables comparables (sin viaje_id ni fechas absolutas, que siempre "driftean").
ref_vars = drift.construir_variables(viajes_ref, estaciones)
act_vars = drift.construir_variables(nuevo, estaciones)
control_vars = ref_vars[(mes_ref == MES_CONTROL).to_numpy()].reset_index(drop=True)
del viajes_ref
ref_vars.head()

## 2. Calidad de datos
Tres vistas complementarias:
- **Descartes de ingesta**: filas inválidas, fuera de periodo o duplicadas que `normalizar_viajes` retiró de cada CSV. El periodo congelado ya viene limpio, así que esta es la única forma de comparar la calidad *cruda*.
- **Chequeos sobre los datos limpios**: duraciones atípicas, estaciones sin nomenclatura, estaciones nuevas o ausentes.
- **Resumen por variable** y el **reporte de Evidently** (sección 3).

In [ ]:
descartes = drift.tabla_descartes(drift.descartes_referencia(), stats_nuevo, MES_NUEVO)
display(descartes)
descartes[['invalidos_pct', 'fuera_periodo_pct', 'duplicados_pct']].plot.bar(
    figsize=(10, 3), title='Descartes de ingesta por mes (% de filas originales)')
plt.ylabel('%'); plt.tight_layout()

In [ ]:
calidad = drift.resumen_calidad(ref_vars, act_vars, estaciones)
display(calidad.round(4))
display(drift.resumen_variables(ref_vars, act_vars).round(4))

## 3. Reporte de calidad con Evidently
`DataSummaryPreset` calcula, para cada variable y en ambos conjuntos, faltantes, valores únicos, mínimo, máximo, cuantiles y distribución. Evidently recibe una **muestra reproducible** (`MUESTRA`, semilla del proyecto); los chequeos de la sección 2 usan todas las filas.

In [ ]:
definicion = DataDefinition(
    numerical_columns=list(drift.VARIABLES_NUMERICAS),
    categorical_columns=list(drift.VARIABLES_CATEGORICAS),
)


def a_dataset(df):
    return Dataset.from_pandas(drift.muestrear(df, MUESTRA), data_definition=definicion)


ds_ref, ds_act, ds_control = a_dataset(ref_vars), a_dataset(act_vars), a_dataset(control_vars)

snap_calidad = Report([DataSummaryPreset()]).run(ds_act, ds_ref)
snap_calidad.save_html(str(DIR_REPORTES / f'calidad_viajes_{MES_NUEVO}_vs_entrenamiento.html'))
snap_calidad

## 4. Data drift por variable
`DataDriftPreset` compara cada variable del mes nuevo contra la referencia. Con más de 1,000 filas, Evidently usa por defecto la **distancia de Wasserstein normalizada** para variables numéricas y la **distancia de Jensen-Shannon** para categóricas; en ambos casos hay drift si la distancia supera el umbral (`UMBRAL_DRIFT`). Se prefieren estas distancias a pruebas como Kolmogorov-Smirnov porque, con cientos de miles de viajes, cualquier diferencia diminuta sería "estadísticamente significativa".

El conjunto se considera con drift cuando al menos `PROPORCION_DRIFT` de las variables lo presentan.

In [ ]:
def reporte_drift(actual, referencia, nombre):
    snap = Report(
        [DataDriftPreset(threshold=UMBRAL_DRIFT, drift_share=PROPORCION_DRIFT)],
        include_tests=True,
    ).run(actual, referencia)
    snap.save_html(str(DIR_REPORTES / f'drift_{nombre}.html'))
    tabla = drift.tabla_drift(snap.dict())
    tabla.to_csv(DIR_REPORTES / f'drift_por_variable_{nombre}.csv')
    return snap, tabla


def grafica_drift(tabla, titulo):
    t = tabla.sort_values('puntaje')
    colores = t['drift'].map({True: 'tab:red', False: 'tab:green'}).fillna('tab:gray')
    ax = t['puntaje'].plot.barh(color=colores, figsize=(8, 4), title=titulo)
    ax.axvline(t['umbral'].iloc[0], ls='--', color='k', label='umbral de drift')
    ax.set_xlabel('Distancia (Wasserstein normalizada / Jensen-Shannon)')
    ax.legend(); plt.tight_layout()


snap_drift, tabla_drift = reporte_drift(ds_act, ds_ref, f'{MES_NUEVO}_vs_entrenamiento_viajes')
print(f"Variables con drift: {int(tabla_drift['drift'].sum())} de {len(tabla_drift)}")
display(tabla_drift)
grafica_drift(tabla_drift, f'Drift por variable · {MES_NUEVO} vs entrenamiento (viajes)')

In [ ]:
snap_drift   # reporte interactivo de Evidently (distribuciones por variable)

## 5. Comparación complementaria con el mismo mes del año anterior
La referencia de entrenamiento mezcla nueve meses. Se compara también con octubre de 2025 para explorar diferencias entre ventanas. La persistencia o desaparición del drift no permite atribuir por sí sola una causa estacional ni demostrar cambios en la relación entre las entradas del modelo y su objetivo.


In [ ]:
snap_control, tabla_control = reporte_drift(ds_act, ds_control, f'{MES_NUEVO}_vs_{MES_CONTROL}')
comparacion = pd.concat(
    {'vs_entrenamiento_viajes': tabla_drift[['puntaje', 'drift']],
     f'vs_{MES_CONTROL}': tabla_control[['puntaje', 'drift']]},
    axis=1,
).sort_values(('vs_entrenamiento_viajes', 'puntaje'), ascending=False)
display(comparacion)
grafica_drift(tabla_control, f'Drift por variable · {MES_NUEVO} vs {MES_CONTROL}')

## 6. Cómo leer los resultados y qué decidir
Este análisis por viaje complementa el reporte principal de entradas de P1.
- **Drift en `duracion_min`, `hora_inicio` o `dia_semana`**: cambió el patrón de uso. Relevante para P1 (demanda por estación y hora), que usa rezagos de 24 y 168 h y el calendario.
- **Drift en `origen_id` / `destino_id`** o estaciones nuevas/ausentes: cambió la red o su uso. Las estaciones nuevas no tienen historia para los rezagos del modelo y las ausentes pueden no ser cero real de demanda.
- **`capacidad_origen`, `latitud_origen`, `longitud_origen`**: se derivan de la nomenclatura de 2026-09; si cambian, lo hace la mezcla de estaciones usadas, no los atributos de cada estación.
- **Faltantes, duraciones atípicas o estaciones sin nomenclatura** en la sección 2: señales para revisar captura, catálogo y plausibilidad; no determinan una causa.
- Un drift detectado es una **señal para revisar**, no una orden de reentrenar: validar contra el desempeño real del modelo sobre el mes nuevo antes de decidir.

Los reportes HTML quedan en `reports/drift/` (pueden pesar varios MB; no es necesario versionarlos). Las conclusiones del equipo van debajo, después de ejecutar con los datos reales.